# FAQ Retrieval Eval

Check ingestion and retrieval via Weaviate.

In [ ]:
import sys
sys.path.append("../")

from dotenv import load_dotenv
from llama_index.core import Settings, StorageContext, VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.weaviate import WeaviateVectorStore

from ingestion.faq.faq_doc_builder import build_llama_documents
from ingestion.faq.faq_heading_chunker import parse_faq_docx
from src.core.transformers import ConditionalFixedChunker
from src.core.utils import ingest_pipeline
from vectorstore.weaviate import client

In [ ]:
load_dotenv(override=True)
embedding_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.embed_model = embedding_model

In [ ]:
sections = parse_faq_docx("../data/processed/faq.docx")
documents = build_llama_documents(sections)
nodes = ingest_pipeline(
    documents,
    transformations=[ConditionalFixedChunker(), embedding_model],
)
len(nodes)

In [ ]:
weaviate_client = client.local()

In [ ]:
if weaviate_client.collections.exists("Faq"):
    weaviate_client.collections.delete("Faq")

vector_store = WeaviateVectorStore(weaviate_client=weaviate_client, index_name="Faq")
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes=nodes, storage_context=storage_context)

In [ ]:
retriever = index.as_retriever(similarity_top_k=3)
results = retriever.retrieve("How to register with an email?")

for node in results:
    print(node.text)
    print(node.metadata)
    print("score:", getattr(node, "score", None))
    print("---")

In [ ]:
weaviate_client.close()